# 05 · Account for every source occurrence

Validation asks whether data satisfies rules. Reconciliation asks whether movement or transformation accounted for it. A rejected order can be perfectly reconciled. Exploded rule failures cannot be used as a rejected-row count.


## Environment
Upload the prepared sample files before the session, then use notebook 00 to check the configured storage. This notebook then runs independently, top to bottom. Spark 3.5 is the target; no Hive catalog is used. Set `BASE_PATH` in the following cell or set `DQ_BASE_PATH` in the driver environment.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)


## Inspect the source
Each JSON line is one order envelope. Preserve the original text and source filename before parsing so even corrupt lines remain accountable.


In [ ]:
ORDER_FIELDS = [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount",
    "status",
    "order_date",
    "shipped_date",
    "cancellation_reason",
    "order_total",
    "seller_id",
    "country",
    "arrival_date",
]
order_schema = T.StructType(
    [T.StructField(c, T.StringType(), True) for c in ORDER_FIELDS]
    + [T.StructField("_corrupt_record", T.StringType(), True)]
)


def read_orders(path):
    # Preserve one source envelope per physical JSON line, including malformed JSON.
    # Source row IDs are materialized before branching; raw_text supports replay.
    raw = (
        spark.read.text(path)
        .withColumnRenamed("value", "raw_text")
        .withColumn("source_file", F.input_file_name())
        .withColumn("source_row_id", F.monotonically_increasing_id())
    )
    parsed = raw.withColumn(
        "parsed",
        F.from_json(
            "raw_text",
            order_schema,
            {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"},
        ),
    )
    return parsed.select("source_row_id", "source_file", "raw_text", "parsed.*").cache()


orders = read_orders(f"{RAW_PATH}/orders")
orders.count()  # materialize once before splitting
customers = spark.read.option("header", True).csv(f"{RAW_PATH}/customers")
products = spark.read.option("header", True).csv(f"{RAW_PATH}/products")
items = spark.read.option("header", True).csv(f"{RAW_PATH}/order_items")
orders.show(30, truncate=False)


## Build accepted and rejected datasets
Keep the two branches disjoint and exhaustive. Malformed envelopes belong to quarantine; they do not vanish before the denominator is computed.


In [ ]:
typed = (
    orders.withColumn("customer_id_clean", F.trim("customer_id"))
    .withColumn("status_clean", F.upper(F.trim("status")))
    .withColumn("qty", F.expr("try_cast(quantity as int)"))
    .withColumn("price", F.expr("try_cast(unit_price as decimal(18,2))"))
    .withColumn("discount_value", F.expr("try_cast(discount as decimal(8,2))"))
    .withColumn("total", F.expr("try_cast(order_total as decimal(18,2))"))
    .withColumn("event_date", F.expr("try_cast(order_date as date)"))
    .withColumn("shipped_on", F.expr("try_cast(shipped_date as date)"))
    .withColumn("arrived_on", F.expr("try_cast(arrival_date as date)"))
)
# Reference tables are deduplicated for membership joins, not as a silent repair.
# A separate customer-key check still exposes duplicate reference records.
customer_keys = (
    customers.select(F.col("customer_id").alias("customer_id_clean"))
    .distinct()
    .withColumn("known_customer", F.lit(True))
)
product_keys = (
    products.select("product_id").distinct().withColumn("known_product", F.lit(True))
)
typed = (
    typed.join(customer_keys, "customer_id_clean", "left")
    .join(product_keys, "product_id", "left")
    .withColumn("key_count", F.count("*").over(Window.partitionBy("order_id")))
)

# A predicate means PASS. NULL is a failure unless the rule explicitly permits it.
rules = [
    (
        "DQ000",
        "parseable",
        "raw_text",
        "Malformed JSON",
        F.col("_corrupt_record").isNull() & F.col("order_id").isNotNull(),
    ),
    (
        "DQ001",
        "customer present",
        "customer_id",
        "Missing or blank customer",
        F.length("customer_id_clean") > 0,
    ),
    (
        "DQ002",
        "positive integer quantity",
        "quantity",
        "Not an integer in 1..1000",
        F.col("qty").between(1, 1000),
    ),
    (
        "DQ003",
        "nonnegative price",
        "unit_price",
        "Invalid or negative price",
        F.col("price") >= 0,
    ),
    (
        "DQ004",
        "allowed status",
        "status",
        "Unknown status",
        F.col("status_clean").isin("CREATED", "PAID", "SHIPPED", "CANCELLED"),
    ),
    (
        "DQ005",
        "unique order key",
        "order_id",
        "Duplicate business key; quarantine all copies",
        F.col("key_count") == 1,
    ),
    (
        "DQ006",
        "event window",
        "order_date",
        "Invalid, future, old or outside daily window",
        F.col("event_date") == F.date_sub(F.to_date(F.lit(PROCESSING_DATE)), 1),
    ),
    (
        "DQ007",
        "known customer",
        "customer_id",
        "Customer reference not found",
        F.coalesce(F.col("known_customer"), F.lit(False)),
    ),
    (
        "DQ008",
        "known product",
        "product_id",
        "Product reference not found",
        F.coalesce(F.col("known_product"), F.lit(False)),
    ),
    (
        "DQ009",
        "discount range",
        "discount",
        "Discount outside 0..100",
        F.col("discount_value").between(0, 100),
    ),
    (
        "BR001",
        "shipment date",
        "shipped_date",
        "SHIPPED requires valid shipped_date",
        (F.col("status_clean") != "SHIPPED") | F.col("shipped_on").isNotNull(),
    ),
    (
        "BR002",
        "cancellation reason",
        "cancellation_reason",
        "CANCELLED requires reason",
        (F.col("status_clean") != "CANCELLED")
        | (F.length(F.trim("cancellation_reason")) > 0),
    ),
    (
        "BR003",
        "nonnegative total",
        "order_total",
        "Invalid or negative order total",
        F.col("total") >= 0,
    ),
    (
        "BR004",
        "order arithmetic",
        "order_total",
        "Header differs from quantity times unit price",
        F.abs(F.col("total") - F.col("qty") * F.col("price"))
        <= F.lit("0.01").cast("decimal(18,2)"),
    ),
]
failure_structs = [
    F.when(
        ~F.coalesce(predicate, F.lit(False)),
        F.struct(
            F.lit(rule_id).alias("dq_rule_id"),
            F.lit(name).alias("dq_rule_name"),
            F.lit(column).alias("dq_column"),
            F.lit(reason).alias("dq_reason"),
        ),
    )
    for rule_id, name, column, reason, predicate in rules
]
scored = (
    typed.withColumn(
        "dq_failures", F.filter(F.array(*failure_structs), lambda x: x.isNotNull())
    )
    .withColumn(
        "dq_status", F.when(F.size("dq_failures") == 0, "VALID").otherwise("INVALID")
    )
    .withColumn("pipeline_run_id", F.lit(RUN_ID))
    .withColumn("processing_timestamp", F.current_timestamp())
    .cache()
)
scored.count()
valid = scored.filter("dq_status = 'VALID'")
rejected = scored.filter("dq_status = 'INVALID'")
valid.select("order_id", "quantity", "status", "dq_status").show(truncate=False)
rejected.select("order_id", "source_row_id", "dq_failures").show(30, truncate=False)


## Counts, quantities, amounts and identities
Control sums cover the castable subset only; invalid numeric strings have no financial value we can invent. Reconcile their counts as a separate check. Decimal arithmetic is used before audit values are rendered as strings.


In [ ]:
accounted = valid.unionByName(rejected)


def numeric_total(df, column):
    return df.agg(F.coalesce(F.sum(column), F.lit(0)).alias("v")).first()["v"]


checks = []


def record_check(name, source, target, tolerance=0.0):
    # Compare native Decimal/integer values before formatting the audit row.
    diff = source - target
    checks.append(
        (
            RUN_ID,
            name,
            str(source),
            str(target),
            str(diff),
            str(tolerance),
            "PASS" if abs(diff) <= tolerance else "FAIL",
        )
    )


record_check("source envelopes", orders.count(), accounted.count())
record_check(
    "quantity (castable subset)",
    numeric_total(typed, "qty"),
    numeric_total(accounted, "qty"),
)
record_check(
    "sales (castable subset)",
    numeric_total(typed, "total"),
    numeric_total(accounted, "total"),
)
record_check(
    "uncastable quantity count",
    typed.filter("qty is null").count(),
    accounted.filter("qty is null").count(),
)
record_check(
    "uncastable total count",
    typed.filter("total is null").count(),
    accounted.filter("total is null").count(),
)
reconciliation = spark.createDataFrame(
    checks,
    "run_id string, check_name string, source_value string, target_value string, difference string, tolerance string, status string",
)
reconciliation.show(truncate=False)
# Physical source IDs detect loss or multiplication of duplicate business keys.
source_ids = orders.select("source_row_id")
target_ids = accounted.select("source_row_id")
assert source_ids.exceptAll(target_ids).count() == 0
assert target_ids.exceptAll(source_ids).count() == 0
source_keys = orders.select("order_id").distinct()
target_keys = accounted.select("order_id").distinct()
source_keys.exceptAll(target_keys).show()
target_keys.exceptAll(source_keys).show()


## Group totals reveal offsetting errors
Compare by business date, seller and country with null-safe joins. A global sum can match while two dates are misallocated. The example below deliberately moves 10 units between days while preserving the global total.


In [ ]:
keys = ["order_date", "seller_id", "country"]
left = (
    typed.groupBy(*keys)
    .agg(F.sum("total").alias("source_total"), F.count("*").alias("source_rows"))
    .alias("s")
)
right = (
    accounted.groupBy(*keys)
    .agg(F.sum("total").alias("target_total"), F.count("*").alias("target_rows"))
    .alias("t")
)
condition = F.lit(True)
for key in keys:
    condition = condition & F.col("s." + key).eqNullSafe(F.col("t." + key))
grouped = left.join(right, condition, "full").select(
    *[F.coalesce(F.col("s." + k), F.col("t." + k)).alias(k) for k in keys],
    "source_total",
    "target_total",
    "source_rows",
    "target_rows"
)
grouped.show(truncate=False)
demo = spark.createDataFrame(
    [("2024-01-01", 100, 110), ("2024-01-02", 200, 190)],
    "business_date string, source int, target int",
)
demo.select(F.sum("source"), F.sum("target")).show()
demo.withColumn("difference", F.col("source") - F.col("target")).show()
# Counts alone also miss a replaced key.
a = spark.createDataFrame([("O1",), ("O2",)], "order_id string")
b = spark.createDataFrame([("O1",), ("O3",)], "order_id string")
assert a.count() == b.count()
a.exceptAll(b).show()
b.exceptAll(a).show()


## Persist and discuss checksums
A canonical row hash can expose changed values, but null representation, column order and type formatting must be agreed. Hash collisions remain possible; a checksum is supporting evidence, not a replacement for count, key and financial reconciliation.


In [ ]:
orders.select(
    "order_id",
    F.sha2(
        F.to_json(F.struct(*ORDER_FIELDS), {"ignoreNullFields": "false"}), 256
    ).alias("row_hash"),
).show(truncate=False)
reconciliation.write.mode("errorifexists").parquet(
    f"{AUDIT_PATH}/reconciliation/{RUN_ID}"
)
